# Notebook 27 — T4b: huấn luyện có đối chứng trên development 5×3

Sau N26 Windows PASS, bắt đầu huấn luyện ba cấu hình trên **a3_dev_n5, 5 task ×3 host, 25 qubit**. Chưa chạy ba validation case ở notebook này.

Mỗi nhánh p1 cap100 rồi p2 INTERP cap100; tổng cap600 objective evaluations. Objective là exact mean-energy; góc tốt nhất theo mean-energy, không theo P_opt. Đọc `N27_README_FIRST.md` trước khi chạy. Dự kiến khoảng một giờ hoặc lâu hơn trên Windows, không phải cam kết runtime. Chỉ chạy một campaign QAOA tại một thời điểm.

In [1]:
from pathlib import Path
import sys
import psutil
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src/domain_wall_training.py').exists()), None)
if root is None: raise RuntimeError('Mở notebook trong repo N26 đã giải nén gói N27.')
if str(root) not in sys.path: sys.path.insert(0, str(root))
from src.domain_wall_training import open_campaign, checked_campaign, run_campaign, summarize, export_campaign
from IPython.display import display, Markdown
print('Repo:', root)
print('Available RAM:', round(psutil.virtual_memory().available/2**30, 2), 'GiB')
print('Guard: available RAM >=4 GiB; Aer CPU 4 threads; 25 qubits.')


Repo: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Available RAM: 14.01 GiB
Guard: available RAM >=4 GiB; Aer CPU 4 threads; 25 qubits.


## 1. Mở campaign và kiểm tra kế hoạch

Lần đầu tạo campaign mới và lưu con trỏ N27_ACTIVE.json. Lần sau dùng lại campaign đó, kể cả khi đã hoàn tất. Không xóa con trỏ để chạy lại nhằm tìm kết quả đẹp. Không sửa source, dataset, package hay protocol trong lúc campaign còn cần tiếp tục.

p1 bắt đầu [beta,gamma]=[0.3,1.0] trong đơn vị Hamiltonian max_abs. p2 lặp beta/gamma từ góc mean-energy tốt nhất của chính p1 của nhánh đó.

In [2]:
folder = open_campaign(root)
campaign = checked_campaign(root, folder)
assert campaign['identity']['software_validation'] is False
assert campaign['identity']['input']['instance_id'] == 'a3_dev_n5'
print('CAMPAIGN:', folder)
print('Scope: development only | objective cap:', sum(s['cap'] for s in campaign['plan']))
for s in campaign['plan']:
    print(s['id'], '| cap', s['cap'], '| depends_on', s['depends_on'])


CAMPAIGN: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_training\t4b_20261025T070339188480Z
Scope: development only | objective cap: 600
standard_x_p1 | cap 100 | depends_on None
standard_x_p2 | cap 100 | depends_on standard_x_p1
valid_x_p1 | cap 100 | depends_on None
valid_x_p2 | cap 100 | depends_on valid_x_p1
valid_dw_p1 | cap 100 | depends_on None
valid_dw_p2 | cap 100 | depends_on valid_dw_p1


## 2. Nhánh standard_x: trạng thái |+⟩ và X mixer

Đây là control được huấn luyện lại với cùng ngân sách như hai nhánh còn lại. Chạy p1 rồi p2, không lấy góc đã học của N23/N24 làm chuẩn bị miễn phí.

Mỗi stage có setup/compile/diagonal trước dòng eval1; khoảng im lặng ban đầu không tự nghĩa là treo. Mỗi10 objective evaluations sẽ in tiến trình. Nếu stage thất bại, chạy ô xuất ZIP cuối và gửi kết quả; không xóa raw/ledger hay tự tăng cap.

In [3]:
status = run_campaign(root, folder, variant='standard_x')
assert all(r['status']=='completed' for r in status['rows'] if r['variant']=='standard_x'), 'Stage chưa hoàn tất; chạy ô export cuối để gửi kiểm tra.'
print('standard_x p1+p2 completed; không đồng nghĩa optimizer đã hội tụ.')


Run standard_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=548.977473
  eval 10/100 | E=228.776184
  eval 20/100 | E=227.694397
  eval 30/100 | E=223.244666
  eval 40/100 | E=206.559839
  eval 50/100 | E=196.022445
  eval 60/100 | E=149.542934
  eval 70/100 | E=111.978200
  eval 80/100 | E=106.017256
  eval 90/100 | E=104.983635
  eval 100/100 | E=104.940713
Run standard_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=115.277799
  eval 10/100 | E=85.775642
  eval 20/100 | E=67.455793
  eval 30/100 | E=66.042431
  eval 40/100 | E=65.782108
  eval 50/100 | E=65.158936
  eval 60/100 | E=64.707254
  eval 70/100 | E=63.538506
  eval 80/100 | E=63.143447
  eval 90/100 | E=62.963710
  eval 100/100 | E=62.836536
standard_x p1+p2 completed; không đồng nghĩa optimizer đã hội tụ.


## 3. Nhánh valid_x: trạng thái đầu domain hợp lệ và X mixer

So với control, thay trạng thái đầu. X mixer vẫn có thể đưa xác suất ra khỏi miền mã hóa hợp lệ. Objective/cost/penalty và ngân sách giữ nguyên.

In [4]:
status = run_campaign(root, folder, variant='valid_x')
assert all(r['status']=='completed' for r in status['rows'] if r['variant']=='valid_x'), 'Stage chưa hoàn tất; chạy ô export cuối để gửi kiểm tra.'
print('valid_x p1+p2 completed.')


Run valid_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=351.375418
  eval 10/100 | E=146.515010
  eval 20/100 | E=120.434539
  eval 30/100 | E=111.809702
  eval 40/100 | E=107.495857
  eval 50/100 | E=103.385951
  eval 60/100 | E=100.805474
  eval 70/100 | E=98.537454
  eval 80/100 | E=98.118999
  eval 90/100 | E=97.980586
  eval 100/100 | E=97.862734
Run valid_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=91.785026
  eval 10/100 | E=86.881995
  eval 20/100 | E=73.135532
  eval 30/100 | E=71.950076
  eval 40/100 | E=71.256598
  eval 50/100 | E=70.741204
  eval 60/100 | E=70.487570
  eval 70/100 | E=70.343939
  eval 80/100 | E=69.936429
  eval 90/100 | E=68.932404
  eval 100/100 | E=68.646592
valid_x p1+p2 completed.


## 4. Nhánh valid_dw: cùng trạng thái đầu hợp lệ và mixer AB

So với valid_x, chỉ thay mixer. AB giữ domain assignment; CPU/RAM/y/slack vẫn do cost xử lý. Kiểm P_domain≈1 sau huấn luyện. Mạch AB là tích có thứ tự B(beta)A(beta), không phải một exponential của A+B.

In [5]:
status = run_campaign(root, folder, variant='valid_dw')
assert all(r['status']=='completed' for r in status['rows'] if r['variant']=='valid_dw'), 'Stage chưa hoàn tất; chạy ô export cuối để gửi kiểm tra.'
print('valid_dw p1+p2 completed.')


Run valid_dw_p1 | 25 qubits | cap 100
  eval 1/100 | E=334.250037
  eval 10/100 | E=144.758917
  eval 20/100 | E=137.309167
  eval 30/100 | E=117.086946
  eval 40/100 | E=109.449695
  eval 50/100 | E=103.671640
  eval 60/100 | E=103.669660
  eval 70/100 | E=103.669638
Run valid_dw_p2 | 25 qubits | cap 100
  eval 1/100 | E=95.535702
  eval 10/100 | E=91.289894
  eval 20/100 | E=73.436340
  eval 30/100 | E=69.474743
  eval 40/100 | E=69.281186
  eval 50/100 | E=69.213726
  eval 60/100 | E=69.192919
  eval 70/100 | E=69.175145
  eval 80/100 | E=69.161989
  eval 90/100 | E=69.148051
  eval 100/100 | E=69.141669
valid_dw p1+p2 completed.


## 5. Đọc kết quả và ngân sách

Các xác suất ở bảng này đã nhân100 thành phần trăm; P_ground giữ dạng xác suất khoa học. P_valid/P_opt giữ y đo được, không sửa placement/slack. p2 là độ sâu chính và phải tính cả chi phí p1 của chính nhánh.

completed nghĩa stage chạy xong và scoring đạt kiểm tra; xem thêm optimizer_stop để biết hội tụ hay hết cap. Đạt cap100 không chứng minh hội tụ. Exact P_opt là số đo chính;1024 shot cuối chỉ minh họa, không dùng để chọn góc hay nhánh thắng.

In [6]:
summary = summarize(root, folder)
rows = ['| Nhánh | p | Trạng thái | Calls objective | E | P_domain (%) | P_valid (%) | P_opt (%) | P_ground |',
        '|---|---:|---|---:|---:|---:|---:|---:|---:|']
for r in summary['rows']:
    if 'mean_energy' not in r:
        rows.append(f"| {r['variant']} | {r['p']} | {r['status']} | {r.get('objective_calls_reserved',0)} | — | — | — | — | — |")
        continue
    rows.append(f"| {r['variant']} | {r['p']} | {r['status']} | {r['objective_calls_reserved']} | {r['mean_energy']:.6f} | {100*r['p_domain']:.6f} | {100*r['p_valid']:.6f} | {100*r['p_opt']:.6f} | {r['p_ground']:.3e} |")
display(Markdown('\n'.join(rows)))
print('Complete:', summary['complete'])
print('Objective calls reserved:', summary['objective_calls_reserved'], '/', summary['objective_cap'])
print('All simulation calls reserved (objective + final scoring + sampling):', summary['simulation_calls_reserved'])
print('Sum stage elapsed seconds (không phải benchmark wall-clock toàn campaign):', summary['elapsed_stage_seconds'])
for r in summary['rows']:
    if r.get('optimizer_stop'): print(r['id'], r['optimizer_stop'])


| Nhánh | p | Trạng thái | Calls objective | E | P_domain (%) | P_valid (%) | P_opt (%) | P_ground |
|---|---:|---|---:|---:|---:|---:|---:|---:|
| standard_x | 1 | completed | 100 | 104.940713 | 44.958369 | 7.019757 | 0.527032 | 2.818e-06 |
| standard_x | 2 | completed | 100 | 62.823613 | 52.716557 | 16.336144 | 0.836108 | 3.148e-05 |
| valid_x | 1 | completed | 100 | 97.862734 | 57.190711 | 8.872609 | 0.609217 | 5.793e-06 |
| valid_x | 2 | completed | 100 | 68.646592 | 28.864125 | 7.330222 | 0.811382 | 3.048e-05 |
| valid_dw | 1 | completed | 70 | 103.669638 | 100.000000 | 11.577191 | 0.447307 | 4.590e-06 |
| valid_dw | 2 | completed | 100 | 69.139422 | 100.000000 | 25.617203 | 1.543284 | 3.540e-05 |

Complete: True
Objective calls reserved: 570 / 600
All simulation calls reserved (objective + final scoring + sampling): 579
Sum stage elapsed seconds (không phải benchmark wall-clock toàn campaign): 3080.1975453999985
standard_x_p1 {'status': 3, 'success': False, 'message': 'Return from COBYLA because the objective function has been evaluated MAXFUN times.', 'nfev': 100}
standard_x_p2 {'status': 3, 'success': False, 'message': 'Return from COBYLA because the objective function has been evaluated MAXFUN times.', 'nfev': 100}
valid_x_p1 {'status': 3, 'success': False, 'message': 'Return from COBYLA because the objective function has been evaluated MAXFUN times.', 'nfev': 100}
valid_x_p2 {'status': 3, 'success': False, 'message': 'Return from COBYLA because the objective function has been evaluated MAXFUN times.', 'nfev': 100}
valid_dw_p1 {'status': 0, 'success': True, 'message': 'Return from COBYLA because the trust region radius reaches its lower bound.', 'nfev': 70}
valid_dw_p2 {'stat

## 6. Xuất kết quả để đánh giá

Chạy ô này cả khi cần gửi một campaign chưa hoàn tất. Nếu kernel chết cứng hoặc hash/runtime/source guard từ chối, giữ nguyên thư mục campaign và gửi lỗi để kiểm tra; không tự xóa khóa hay khởi tạo lại.

Sau khi có N27 Windows mới đánh giá T4b và quyết định chuyển sang ba case kiểm chứng với nguyên cấu hình. Không suy kết luận chung từ riêng development.

In [7]:
archive = export_campaign(root, folder)
print('GỬI LẠI ZIP:', archive)
print('Save notebook 27 đã chạy và gửi kèm ZIP này.')
print('Không cần gửi toàn bộ repo. Giữ nguyên thư mục campaign để truy vết.')


GỬI LẠI ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_training\t4b_20261025T070339188480Z.zip
Save notebook 27 đã chạy và gửi kèm ZIP này.
Không cần gửi toàn bộ repo. Giữ nguyên thư mục campaign để truy vết.
